# 05 - Casos interesantes

Este notebook busca automáticamente funciones útiles para comentar en la memoria.

No se trata solo de buscar lo más complejo, sino de encontrar ejemplos donde las herramientas coinciden o discrepan.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEVEL_ORDER = {"A1": 1, "A2": 2, "B1": 3, "B2": 4, "C1": 5, "C2": 6}
LEVEL_ORDER_INV = {v: k for k, v in LEVEL_ORDER.items()}
RADON_RANK_ORDER = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6}
RADON_RANK_ORDER_INV = {v: k for k, v in RADON_RANK_ORDER.items()}

def clean_file_name(path):
    """Devuelve un nombre de fichero comparable entre herramientas."""
    return Path(str(path)).name

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def detect_json_kind(path):
    """Intenta detectar si un JSON parece de Radon o de PyCEFR."""
    try:
        data = load_json(path)
    except Exception:
        return "unknown"
    # Buscar un registro de ejemplo dentro del árbol project/file/[records]
    for project, files in data.items():
        if not isinstance(files, dict):
            continue
        for file_path, records in files.items():
            if isinstance(records, list) and records:
                rec = records[0]
                if isinstance(rec, dict):
                    if {"Class", "Start Line", "End Line", "Level"}.issubset(set(rec.keys())):
                        return "pycefr"
                    if {"type", "rank", "complexity", "lineno", "endline"}.issubset(set(rec.keys())):
                        return "radon"
    return "unknown"


def cross_radon_pycefr(df_radon, df_pycefr):
    """Cruza cada función Radon con los constructos PyCEFR incluidos en su rango de líneas."""
    rows = []
    if df_radon.empty or df_pycefr.empty:
        return pd.DataFrame(rows)
    for _, fn in df_radon.iterrows():
        subset = df_pycefr[
            (df_pycefr["case"] == fn["case"]) &
            (df_pycefr["file_name"] == fn["file_name"]) &
            (df_pycefr["start_line"] >= fn["lineno"]) &
            (df_pycefr["end_line"] <= fn["endline"])
        ]
        level_counts = subset["level"].value_counts().to_dict() if not subset.empty else {}
        class_counts = subset["class"].value_counts().head(10).to_dict() if not subset.empty else {}
        max_level_num = subset["level_num"].max() if not subset.empty else np.nan
        rows.append({
            "case": fn["case"],
            "file": fn["file"],
            "file_name": fn["file_name"],
            "function": fn["name"],
            "parent": fn.get("parent"),
            "lineno": fn["lineno"],
            "endline": fn["endline"],
            "complexity": fn["complexity"],
            "rank": fn["rank"],
            "rank_num": fn["rank_num"],
            "n_pycefr_constructs": len(subset),
            "n_pycefr_classes": subset["class"].nunique() if not subset.empty else 0,
            "max_level_num": max_level_num,
            "max_level": LEVEL_ORDER_INV.get(int(max_level_num)) if pd.notna(max_level_num) else None,
            "mean_level_num": subset["level_num"].mean() if not subset.empty else np.nan,
            "level_counts": json.dumps(level_counts, ensure_ascii=False),
            "top_classes": json.dumps(class_counts, ensure_ascii=False),
        })
    return pd.DataFrame(rows)

def label_interesting_cases(cross):
    df = cross.copy()
    if df.empty:
        df["case_type"] = []
        return df
    conditions = []
    for _, row in df.iterrows():
        labels = []
        if row.get("rank_num", 0) >= 2 and row.get("max_level_num", 0) <= 2:
            labels.append("Radon alto / PyCEFR bajo")
        if row.get("rank_num", 0) <= 1 and row.get("max_level_num", 0) >= 5:
            labels.append("PyCEFR alto / Radon bajo")
        if row.get("rank_num", 0) >= 2 and row.get("max_level_num", 0) >= 4:
            labels.append("Coincidencia en dificultad alta")
        if row.get("n_pycefr_constructs", 0) >= 25 and row.get("max_level_num", 0) <= 2:
            labels.append("Muchos constructos simples acumulados")
        if not labels:
            labels.append("Sin patrón destacado")
        conditions.append("; ".join(labels))
    df["case_type"] = conditions
    return df

In [2]:
cross = pd.read_csv(OUTPUT_DIR / "04_radon_pycefr_cross_global.csv")
interesting = label_interesting_cases(cross)
interesting.head()

,case,file,file_name,function,parent,lineno,endline,complexity,rank,rank_num,n_pycefr_constructs,n_pycefr_classes,max_level_num,max_level,mean_level_num,level_counts,top_classes,case_type
0,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,test.py,test_for_return,NaN,21,26,4,A,1,147,19,5.0,C1,2.034014,"{""A2"": 78, ""A1"": 42, ""B2"": 18, ""B1"": 8, ""C1"": 1}","{""Simple Atributte"": 62, ""Simple Assignment"": ...",PyCEFR alto / Radon bajo
1,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,test.py,test_function_exists,NaN,12,17,2,A,1,134,16,4.0,B2,1.985075,"{""A2"": 65, ""A1"": 39, ""B1"": 23, ""B2"": 7}","{""Simple Atributte"": 52, ""'raise' exception"": ...",Sin patrón destacado
2,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,test.py,test_for_type_random,NaN,31,35,2,A,1,68,18,5.0,C1,1.838235,"{""A2"": 33, ""A1"": 27, ""B2"": 6, ""C1"": 1, ""B1"": 1}","{""Simple Atributte"": 23, ""Simple Assignment"": ...",PyCEFR alto / Radon bajo
3,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,test.py,test_function_called_for,NaN,41,41,2,A,1,8,3,4.0,B2,3.000000,"{""B2"": 4, ""A2"": 4}","{""'assert' exception"": 4, ""Simple Atributte"": ...",Sin patrón destacado
4,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,generate_random,NaN,4,6,1,A,1,31,8,2.0,A2,1.193548,"{""A1"": 25, ""A2"": 6}","{""Simple Assignment"": 10, ""Print"": 9, ""Simple ...",Muchos constructos simples acumulados


In [3]:
interesting["case_type"].value_counts()

case_type
Sin patrón destacado                                               63
PyCEFR alto / Radon bajo                                           32
Muchos constructos simples acumulados                              12
Radon alto / PyCEFR bajo; Muchos constructos simples acumulados     1
Coincidencia en dificultad alta                                     1
Name: count, dtype: int64

In [4]:
selected = interesting[interesting["case_type"] != "Sin patrón destacado"].copy()
selected = selected.sort_values(["case_type", "complexity", "max_level_num", "n_pycefr_constructs"], ascending=[True, False, False, False])
selected.head(50)

,case,file,file_name,function,parent,lineno,endline,complexity,rank,rank_num,n_pycefr_constructs,n_pycefr_classes,max_level_num,max_level,mean_level_num,level_counts,top_classes,case_type
91,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,test.py,test_for_function_output,NaN,29,29,6,B,2,17,5,5.0,C1,1.941176,"{""A2"": 9, ""A1"": 6, ""C1"": 1, ""B2"": 1}","{""Simple Atributte"": 8, ""Simple Assignment"": 6...",Coincidencia en dificultad alta
76,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,sing,NaN,2,11,4,A,1,81,12,2.0,A2,1.148148,"{""A1"": 69, ""A2"": 12}","{""Print"": 20, ""Simple Assignment"": 19, ""Return...",Muchos constructos simples acumulados
92,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,number_of_bottles,NaN,2,8,2,A,1,68,11,2.0,A2,1.161765,"{""A1"": 57, ""A2"": 11}","{""Simple Assignment"": 19, ""Print"": 17, ""Return...",Muchos constructos simples acumulados
38,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,standards_maker,NaN,1,4,2,A,1,46,12,2.0,A2,1.217391,"{""A1"": 36, ""A2"": 10}","{""Simple Assignment"": 12, ""Print"": 7, ""Import""...",Muchos constructos simples acumulados
63,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,start_counting,NaN,1,4,2,A,1,46,12,2.0,A2,1.217391,"{""A1"": 36, ""A2"": 10}","{""Simple Assignment"": 12, ""Print"": 7, ""Import""...",Muchos constructos simples acumulados
87,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,get_color,NaN,3,15,1,A,1,86,14,2.0,A2,1.162791,"{""A1"": 72, ""A2"": 14}","{""Print"": 21, ""Simple Assignment"": 19, ""Return...",Muchos constructos simples acumulados
20,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,get_randomInt,NaN,3,6,1,A,1,49,11,2.0,A2,1.183673,"{""A1"": 40, ""A2"": 9}","{""Simple Assignment"": 16, ""Print"": 11, ""Simple...",Muchos constructos simples acumulados
52,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,get_randomInt,NaN,3,6,1,A,1,49,11,2.0,A2,1.183673,"{""A1"": 40, ""A2"": 9}","{""Simple Assignment"": 16, ""Print"": 11, ""Simple...",Muchos constructos simples acumulados
88,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,app.py,get_color,NaN,3,15,1,A,1,33,8,2.0,A2,1.151515,"{""A1"": 28, ""A2"": 5}","{""Simple Assignment"": 11, ""Print"": 6, ""Return""...",Muchos constructos simples acumulados
4,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,generate_random,NaN,4,6,1,A,1,31,8,2.0,A2,1.193548,"{""A1"": 25, ""A2"": 6}","{""Simple Assignment"": 10, ""Print"": 9, ""Simple ...",Muchos constructos simples acumulados


In [5]:
# Top de ejemplos para revisar a mano
cols = ["case", "file_name", "function", "lineno", "endline", "complexity", "rank", "max_level", "n_pycefr_constructs", "case_type", "top_classes"]
examples = selected[cols].head(30)
examples

,case,file_name,function,lineno,endline,complexity,rank,max_level,n_pycefr_constructs,case_type,top_classes
91,python-beginner-programming-exercises,test.py,test_for_function_output,29,29,6,B,C1,17,Coincidencia en dificultad alta,"{""Simple Atributte"": 8, ""Simple Assignment"": 6..."
76,python-beginner-programming-exercises,solution.hide.py,sing,2,11,4,A,A2,81,Muchos constructos simples acumulados,"{""Print"": 20, ""Simple Assignment"": 19, ""Return..."
92,python-beginner-programming-exercises,solution.hide.py,number_of_bottles,2,8,2,A,A2,68,Muchos constructos simples acumulados,"{""Simple Assignment"": 19, ""Print"": 17, ""Return..."
38,python-beginner-programming-exercises,solution.hide.py,standards_maker,1,4,2,A,A2,46,Muchos constructos simples acumulados,"{""Simple Assignment"": 12, ""Print"": 7, ""Import""..."
63,python-beginner-programming-exercises,solution.hide.py,start_counting,1,4,2,A,A2,46,Muchos constructos simples acumulados,"{""Simple Assignment"": 12, ""Print"": 7, ""Import""..."
87,python-beginner-programming-exercises,solution.hide.py,get_color,3,15,1,A,A2,86,Muchos constructos simples acumulados,"{""Print"": 21, ""Simple Assignment"": 19, ""Return..."
20,python-beginner-programming-exercises,solution.hide.py,get_randomInt,3,6,1,A,A2,49,Muchos constructos simples acumulados,"{""Simple Assignment"": 16, ""Print"": 11, ""Simple..."
52,python-beginner-programming-exercises,solution.hide.py,get_randomInt,3,6,1,A,A2,49,Muchos constructos simples acumulados,"{""Simple Assignment"": 16, ""Print"": 11, ""Simple..."
88,python-beginner-programming-exercises,app.py,get_color,3,15,1,A,A2,33,Muchos constructos simples acumulados,"{""Simple Assignment"": 11, ""Print"": 6, ""Return""..."
4,python-beginner-programming-exercises,solution.hide.py,generate_random,4,6,1,A,A2,31,Muchos constructos simples acumulados,"{""Simple Assignment"": 10, ""Print"": 9, ""Simple ..."


In [6]:
interesting.to_csv(OUTPUT_DIR / "05_casos_interesantes_global.csv", index=False)
examples.to_csv(OUTPUT_DIR / "05_ejemplos_para_memoria.csv", index=False)
print("Guardado:", OUTPUT_DIR / "05_ejemplos_para_memoria.csv")

Guardado: /home/juan/Documents/Analisis-CC-PyCEFR/outputs/05_ejemplos_para_memoria.csv


## Comentario para la memoria

Los casos interesantes permiten convertir los resultados numéricos en ejemplos interpretables. En particular, son útiles los casos en los que Radon asigna alta complejidad pero PyCEFR solo encuentra constructos básicos, y los casos inversos, donde PyCEFR detecta constructos avanzados aunque la complejidad ciclomática sea baja. Estos contrastes ayudan a explicar que ambas herramientas capturan dimensiones distintas de la dificultad.